# Análise Final Parametrizada (Sem Moodle)

> Notebook parametrizado para executar a análise final permitindo escolher entre dois modos:
- incremental (início, fim e passo configuráveis, ex.: de 100 em 100);
- amostras (N pontos entre N_MIN e V_MAX, em escala linear ou logarítmica).

Instâncias suportadas: '10k' e '1m'.

---

In [ ]:
# Imports necessários
from typing import Dict, List, Tuple
from collections import deque
from math import inf
import heapq
import os
import matplotlib.pyplot as plt
import numpy as np

## Configurações Iniciais

In [ ]:
# Configurações
ORIGEM = 0
SEED = 42

# Caminhos dos arquivos (sem Moodle)
caminho_10k = r'D:\OneDrive\Pessoais\Doutorado\Cefet\022025\Teoria de Grafos\10000.txt'
caminho_1m  = r'D:\OneDrive\Pessoais\Doutorado\Cefet\022025\Teoria de Grafos\1000000.txt'

# Instância ativa: aceite '10k' ou '1m' (case-insensitive)
INSTANCIA = '10k'  # ou '1m'

# Modo de geração de tamanhos:
# - 'incremental': gera de INICIO a FIM com passo configurável (ex.: de 100 em 100)
# - 'amostras': gera N_AMOSTRAS valores entre N_MIN e V_MAX (escala 'lin' ou 'log')
MODO_TAMANHOS = 'amostras'   # 'incremental' ou 'amostras'

# Configurações do modo 'incremental'
INICIO_INCREMENTAL = None    # se None, usa N_MIN da instância
FIM_INCREMENTAL    = None    # se None, usa V_MAX da instância
PASSO_INCREMENTAL  = 200     # ex.: 100 para "de 100 em 100"

# Configurações do modo 'amostras'
N_AMOSTRAS = 50              # quantidade de amostras quando MODO_TAMANHOS='amostras'
ESCALA_AMOSTRAS = 'lin'      # 'lin' ou 'log'

# Seleção de caminho e parâmetros por instância
inst_norm = INSTANCIA.lower()
if inst_norm == '10k':
    caminho = caminho_10k
    V_MAX = 10_000
    N_MIN = 10
    inst_suf = '10k'
elif inst_norm == '1m':
    caminho = caminho_1m
    V_MAX = 1_000_000
    N_MIN = 1_000  # ponto inicial mais razoável para reduzir custo
    inst_suf = '1m'
else:
    raise ValueError("INSTANCIA inválida. Use '10k' ou '1m'.")

# Construção da lista de tamanhos conforme o modo
if MODO_TAMANHOS == 'incremental':
    inicio = INICIO_INCREMENTAL if INICIO_INCREMENTAL is not None else N_MIN
    fim    = FIM_INCREMENTAL    if FIM_INCREMENTAL    is not None else V_MAX
    inicio = max(2, int(inicio))
    fim    = int(fim)
    passo  = max(1, int(PASSO_INCREMENTAL))
    tamanhos = list(range(inicio, fim + 1, passo))
    if tamanhos and tamanhos[-1] != fim and fim > tamanhos[-1]:
        tamanhos.append(fim)
else:  # 'amostras'
    N_MIN = max(2, N_MIN)
    if ESCALA_AMOSTRAS == 'log':
        xs = np.logspace(np.log10(N_MIN), np.log10(V_MAX), num=max(2, N_AMOSTRAS))
    else:  # 'lin'
        xs = np.linspace(N_MIN, V_MAX, num=max(2, N_AMOSTRAS))
    tamanhos = sorted({int(round(x)) for x in xs if N_MIN <= x <= V_MAX})

print('Verificando arquivos:')
print(f"  10k: {'✓ Encontrado' if os.path.exists(caminho_10k) else '✗ Não encontrado'}")
print(f"  1M : {'✓ Encontrado' if os.path.exists(caminho_1m) else '✗ Não encontrado'}")

print(f"\nInstância ativa: {INSTANCIA} -> {caminho}")
if MODO_TAMANHOS == 'incremental':
    print(f"Modo tamanhos: incremental (início={inicio:,}, fim={fim:,}, passo={passo:,})")
else:
    print(f"Modo tamanhos: amostras ({ESCALA_AMOSTRAS})")
print(f'Total de testes: {len(tamanhos)}')
print(f'Primeiros: {tamanhos[:10]}')
print(f'Últimos: {tamanhos[-10:]}')

## Funções Auxiliares

In [ ]:
def carregar_grafo_ewd(caminho: str, max_vertices: int = None):
    """Carrega grafo em formato EWD, limitado a max_vertices."""
    with open(caminho, 'r', encoding='utf-8') as f:
        V = int(f.readline().strip())
        E = int(f.readline().strip())
        
        if max_vertices is not None:
            V = min(V, max_vertices)
        
        adj = {i: [] for i in range(V)}
        num_arestas = 0
        
        for _ in range(E):
            linha = f.readline().strip()
            if not linha:
                break
            u, v, peso = linha.split()[:3]
            u, v = int(u), int(v)
            
            if u < V and v < V:
                adj[u].append((v, float(peso)))
                num_arestas += 1
    
    return adj, num_arestas

print('✓ Função de carregamento criada')

---
# PARTE 1 — ALGORITMO DE DIJKSTRA
---
## Implementação

In [ ]:
def dijkstra(adj: Dict[int, List[Tuple[int, float]]], origem: int = 0):
    n = len(adj)
    dist = [inf] * n
    parent = [-1] * n
    dist[origem] = 0
    heap = [(0, origem)]
    comparacoes = 0
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            continue
        for v, peso in adj[u]:
            comparacoes += 1
            nova = dist[u] + peso
            if nova < dist[v]:
                dist[v] = nova
                parent[v] = u
                heapq.heappush(heap, (nova, v))
    return dist, parent, comparacoes

print('✓ Dijkstra implementado')

## Testes Incrementais com Dijkstra

In [ ]:
resultados_dijkstra = {'tamanhos': [], 'arestas': [], 'comparacoes': [], 'alcancados': []}

if os.path.exists(caminho):
    print('='*70)
    print(f'PARTE 1: TESTES INCREMENTAIS - DIJKSTRA ({INSTANCIA})')
    print('='*70)
    for i, tam in enumerate(tamanhos, 1):
        grafo, num_arestas = carregar_grafo_ewd(caminho, max_vertices=tam)
        dist, parent, comp = dijkstra(grafo, origem=ORIGEM)
        alc = sum(1 for d in dist if d != inf)
        resultados_dijkstra['tamanhos'].append(tam)
        resultados_dijkstra['arestas'].append(num_arestas)
        resultados_dijkstra['comparacoes'].append(comp)
        resultados_dijkstra['alcancados'].append(alc)
        if i % 20 == 0 or i == len(tamanhos):
            print(f'  [{i}/{len(tamanhos)}] n={tam:,}: {comp:,} comp, {alc:,} vértices')
    print('\n✓ Testes Dijkstra concluídos!')
else:
    print('✗ Arquivo não encontrado!')

## Salvar Resultados - Dijkstra

In [ ]:
if resultados_dijkstra['tamanhos']:
    os.makedirs('resultados/analise_esparsos', exist_ok=True)
    with open(f'resultados/analise_esparsos/parte1_dijkstra_{inst_suf}.txt', 'w', encoding='utf-8') as f:
        f.write(f'PARTE 1 — ALGORITMO DE DIJKSTRA (Sem Moodle) — Instância {INSTANCIA}\n')
        f.write('='*80 + '\n\n')
        f.write(f'{"Vértices":<12}{"Arestas":<12}{"Comparações":<16}{"Alcançados":<14}{"% Alcançados":<14}\n')
        f.write('-'*80 + '\n')
        for i in range(len(resultados_dijkstra['tamanhos'])):
            tam = resultados_dijkstra['tamanhos'][i]
            arestas = resultados_dijkstra['arestas'][i]
            comp = resultados_dijkstra['comparacoes'][i]
            alc = resultados_dijkstra['alcancados'][i]
            perc = (alc / tam * 100) if tam > 0 else 0
            f.write(f'{tam:<12,}{arestas:<12,}{comp:<16,}{alc:<14,}{perc:<14.2f}%\n')
    print(f'✓ Resultados salvos: resultados/analise_esparsos/parte1_dijkstra_{inst_suf}.txt')

## Gráficos - Dijkstra

In [ ]:
if resultados_dijkstra['tamanhos']:
    os.makedirs('resultados/analise_esparsos/img', exist_ok=True)
    plt.figure(figsize=(8, 5))
    plt.plot(resultados_dijkstra['tamanhos'], resultados_dijkstra['comparacoes'], marker='o', linewidth=2, markersize=4, color='steelblue')
    plt.xlabel('Número de vértices')
    plt.ylabel('Número de comparações')
    plt.title('Dijkstra - Comparações vs Vértices')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(f'resultados/analise_esparsos/img/parte1_dijkstra_comp_{inst_suf}.png', dpi=150)
    plt.show()
    plt.figure(figsize=(8, 5))
    plt.plot(resultados_dijkstra['tamanhos'], resultados_dijkstra['alcancados'], marker='o', linewidth=2, markersize=4, color='green')
    plt.xlabel('Número de vértices')
    plt.ylabel('Vértices alcançados')
    plt.title('Dijkstra - Vértices Alcançados')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(f'resultados/analise_esparsos/img/parte1_dijkstra_alcancados_{inst_suf}.png', dpi=150)
    plt.show()
else:
    print('✗ Resultados Dijkstra indisponíveis para gráficos')

---
# PARTE 2 — HEURÍSTICA GULOSA
---
## Implementação

In [ ]:
def heuristica_gulosa(adj: Dict[int, List[Tuple[int, float]]], origem: int = 0):
    n = len(adj)
    dist = [inf] * n
    parent = [-1] * n
    dist[origem] = 0
    visited = {origem}
    fila = deque([origem])
    comparacoes = 0
    while fila:
        u = fila.popleft()
        melhor_v, melhor_custo = None, inf
        for v, peso in adj[u]:
            comparacoes += 1
            if v in visited:
                continue
            custo = dist[u] + peso
            if custo < dist[v] and custo < melhor_custo:
                melhor_custo, melhor_v = custo, v
        if melhor_v is not None:
            dist[melhor_v] = melhor_custo
            parent[melhor_v] = u
            visited.add(melhor_v)
            fila.append(melhor_v)
    return dist, parent, comparacoes

print('✓ Heurística Gulosa implementada')

## Testes Incrementais com Heurística Gulosa

In [ ]:
resultados_gulosa = {'tamanhos': [], 'arestas': [], 'comparacoes': [], 'alcancados': []}

if os.path.exists(caminho):
    print('='*70)
    print(f'PARTE 2: TESTES INCREMENTAIS - HEURÍSTICA GULOSA ({INSTANCIA})')
    print('='*70)
    for i, tam in enumerate(tamanhos, 1):
        grafo, num_arestas = carregar_grafo_ewd(caminho, max_vertices=tam)
        dist, parent, comp = heuristica_gulosa(grafo, origem=ORIGEM)
        alc = sum(1 for d in dist if d != inf)
        resultados_gulosa['tamanhos'].append(tam)
        resultados_gulosa['arestas'].append(num_arestas)
        resultados_gulosa['comparacoes'].append(comp)
        resultados_gulosa['alcancados'].append(alc)
        if i % 20 == 0 or i == len(tamanhos):
            print(f'  [{i}/{len(tamanhos)}] n={tam:,}: {comp:,} comp, {alc:,} vértices')
    print('\n✓ Testes Gulosa concluídos!')
else:
    print('✗ Arquivo não encontrado!')

## Salvar Resultados - Heurística Gulosa

In [ ]:
if resultados_gulosa['tamanhos']:
    os.makedirs('resultados/analise_esparsos', exist_ok=True)
    with open(f'resultados/analise_esparsos/parte2_gulosa_{inst_suf}.txt', 'w', encoding='utf-8') as f:
        f.write(f'PARTE 2 — HEURÍSTICA GULOSA (Sem Moodle) — Instância {INSTANCIA}\n')
        f.write('='*80 + '\n\n')
        f.write(f'{"Vértices":<12}{"Arestas":<12}{"Comparações":<16}{"Alcançados":<14}{"% Alcançados":<14}\n')
        f.write('-'*80 + '\n')
        for i in range(len(resultados_gulosa['tamanhos'])):
            tam = resultados_gulosa['tamanhos'][i]
            arestas = resultados_gulosa['arestas'][i]
            comp = resultados_gulosa['comparacoes'][i]
            alc = resultados_gulosa['alcancados'][i]
            perc = (alc / tam * 100) if tam > 0 else 0
            f.write(f'{tam:<12,}{arestas:<12,}{comp:<16,}{alc:<14,}{perc:<14.2f}%\n')
    print(f'✓ Resultados salvos: resultados/analise_esparsos/parte2_gulosa_{inst_suf}.txt')

## Gráficos - Heurística Gulosa

In [ ]:
if resultados_gulosa['tamanhos']:
    os.makedirs('resultados/analise_esparsos/img', exist_ok=True)
    plt.figure(figsize=(8, 5))
    plt.plot(resultados_gulosa['tamanhos'], resultados_gulosa['comparacoes'], marker='s', linewidth=2, markersize=4, color='orange')
    plt.xlabel('Número de vértices')
    plt.ylabel('Número de comparações')
    plt.title('Gulosa - Comparações vs Vértices')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(f'resultados/analise_esparsos/img/parte2_gulosa_comp_{inst_suf}.png', dpi=150)
    plt.show()
    plt.figure(figsize=(8, 5))
    plt.plot(resultados_gulosa['tamanhos'], resultados_gulosa['alcancados'], marker='s', linewidth=2, markersize=4, color='red')
    plt.xlabel('Número de vértices')
    plt.ylabel('Vértices alcançados')
    plt.title('Gulosa - Vértices Alcançados')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(f'resultados/analise_esparsos/img/parte2_gulosa_alcancados_{inst_suf}.png', dpi=150)
    plt.show()
else:
    print('✗ Resultados Gulosa indisponíveis para gráficos')

---
# COMPARAÇÃO ENTRE OS ALGORITMOS
---

## Tabela Comparativa e Gráficos

In [ ]:
if resultados_dijkstra['tamanhos'] and resultados_gulosa['tamanhos']:
    with open(f'resultados/analise_esparsos/comparacao_{inst_suf}.txt', 'w', encoding='utf-8') as f:
        f.write(f'COMPARAÇÃO: DIJKSTRA vs HEURÍSTICA GULOSA (Sem Moodle) — Instância {INSTANCIA}\n')
        f.write('='*120 + '\n\n')
        f.write(f'{"Vértices":<10}{"Arestas":<10}{"Dij Comp":<14}{"Dij Alc":<12}{"Gul Comp":<14}{"Gul Alc":<12}{"Red Comp%":<12}{"Perda Alc%":<14}\n')
        f.write('-'*120 + '\n')
        for i in range(len(resultados_dijkstra['tamanhos'])):
            tam = resultados_dijkstra['tamanhos'][i]
            arestas = resultados_dijkstra['arestas'][i]
            d_comp = resultados_dijkstra['comparacoes'][i]
            d_alc = resultados_dijkstra['alcancados'][i]
            g_comp = resultados_gulosa['comparacoes'][i]
            g_alc = resultados_gulosa['alcancados'][i]
            red_comp = (1 - g_comp/d_comp)*100 if d_comp > 0 else 0
            perda_alc = (1 - g_alc/d_alc)*100 if d_alc > 0 else 0
            f.write(f'{tam:<10,}{arestas:<10,}{d_comp:<14,}{d_alc:<12,}{g_comp:<14,}{g_alc:<12,}{red_comp:<12.2f}%{perda_alc:<14.2f}%\n')
    os.makedirs('resultados/analise_esparsos/img', exist_ok=True)
    # Comparações linear
    plt.figure(figsize=(8,5))
    plt.plot(resultados_dijkstra['tamanhos'], resultados_dijkstra['comparacoes'], marker='o', linewidth=2, markersize=4, color='steelblue', label='Dijkstra')
    plt.plot(resultados_gulosa['tamanhos'], resultados_gulosa['comparacoes'], marker='s', linewidth=2, markersize=4, color='orange', label='Gulosa')
    plt.xlabel('Número de vértices'); plt.ylabel('Comparações'); plt.title('Comparações: Dijkstra vs Gulosa'); plt.legend(); plt.grid(True, linestyle='--', alpha=0.6); plt.tight_layout()
    plt.savefig(f'resultados/analise_esparsos/img/comparacao_comp_{inst_suf}.png', dpi=150); plt.show()
    # Comparações log-log
    plt.figure(figsize=(8,5))
    plt.plot(resultados_dijkstra['tamanhos'], resultados_dijkstra['comparacoes'], marker='o', linewidth=2, markersize=4, color='steelblue', label='Dijkstra')
    plt.plot(resultados_gulosa['tamanhos'], resultados_gulosa['comparacoes'], marker='s', linewidth=2, markersize=4, color='orange', label='Gulosa')
    plt.xscale('log'); plt.yscale('log')
    plt.xlabel('Número de vértices (log)'); plt.ylabel('Comparações (log)'); plt.title('Comparações (log-log): Dijkstra vs Gulosa'); plt.legend(); plt.grid(True, which='both', linestyle='--', alpha=0.6); plt.tight_layout()
    plt.savefig(f'resultados/analise_esparsos/img/comparacao_comp_log_{inst_suf}.png', dpi=150); plt.show()
    # Alcançados
    plt.figure(figsize=(8,5))
    plt.plot(resultados_dijkstra['tamanhos'], resultados_dijkstra['alcancados'], marker='o', linewidth=2, markersize=4, color='green', label='Dijkstra')
    plt.plot(resultados_gulosa['tamanhos'], resultados_gulosa['alcancados'], marker='s', linewidth=2, markersize=4, color='red', label='Gulosa')
    plt.xlabel('Número de vértices'); plt.ylabel('Vértices alcançados'); plt.title('Vértices Alcançados: Dijkstra vs Gulosa'); plt.legend(); plt.grid(True, linestyle='--', alpha=0.6); plt.tight_layout()
    plt.savefig(f'resultados/analise_esparsos/img/comparacao_alcancados_{inst_suf}.png', dpi=150); plt.show()
    # Redução de comparações
    reducao_comp = [(1 - g/d)*100 if d > 0 else 0 for g,d in zip(resultados_gulosa['comparacoes'], resultados_dijkstra['comparacoes'])]
    plt.figure(figsize=(8,5))
    plt.plot(resultados_dijkstra['tamanhos'], reducao_comp, marker='o', linewidth=2, markersize=4, color='purple')
    plt.xlabel('Número de vértices'); plt.ylabel('Redução de comparações (%)'); plt.title('Redução de Comparações pela Gulosa'); plt.grid(True, linestyle='--', alpha=0.6); plt.axhline(y=0, color='black', linestyle='-', alpha=0.3); plt.tight_layout()
    plt.savefig(f'resultados/analise_esparsos/img/reducao_comparacoes_{inst_suf}.png', dpi=150); plt.show()
    # Taxa de alcance
    taxa_alc = [(g/d)*100 if d > 0 else 0 for g,d in zip(resultados_gulosa['alcancados'], resultados_dijkstra['alcancados'])]
    plt.figure(figsize=(8,5))
    plt.plot(resultados_dijkstra['tamanhos'], taxa_alc, marker='o', linewidth=2, markersize=4, color='crimson')
    plt.xlabel('Número de vértices'); plt.ylabel('% de vértices alcançados'); plt.title('Taxa de Alcance da Gulosa'); plt.grid(True, linestyle='--', alpha=0.6); plt.axhline(y=100, color='green', linestyle='--', alpha=0.5, label='100% (ideal)'); plt.legend(); plt.tight_layout()
    plt.savefig(f'resultados/analise_esparsos/img/taxa_alcance_gulosa_{inst_suf}.png', dpi=150); plt.show()
else:
    print('✗ Resultados insuficientes para comparação')